# Category Insights - Load Data to Unity Catalog

This notebook loads the healthcare supply chain category insights data into a Unity Catalog Delta table.

**Source:** Unity Catalog Volume  
**Target Table:** `archana_krish_fe_dsa.vizient_deep_dive.category_insights_delta`  
**Volume:** `archana_krish_fe_dsa.vizient_deep_dive.data_files`

In [ ]:
# Configuration
CATALOG = "archana_krish_fe_dsa"
SCHEMA = "vizient_deep_dive"
VOLUME = "data_files"
CSV_FILE = "category_insights_data.csv"
TARGET_TABLE = f"{CATALOG}.{SCHEMA}.category_insights_delta"
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/{CSV_FILE}"

print(f"📋 Configuration:")
print(f"   Catalog: {CATALOG}")
print(f"   Schema: {SCHEMA}")
print(f"   Volume: {VOLUME}")
print(f"   CSV Path: {VOLUME_PATH}")
print(f"   Target Table: {TARGET_TABLE}")

## Step 1: Create Unity Catalog Schema and Volume (if not exists)

In [ ]:
# Create schema if it doesn't exist
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
print(f"✅ Schema {CATALOG}.{SCHEMA} ready")

# Create volume if it doesn't exist
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")
print(f"✅ Volume {CATALOG}.{SCHEMA}.{VOLUME} ready")
print(f"")
print(f"📁 Volume location: {VOLUME_PATH}")
print(f"")
print(f"⚠️  IMPORTANT: Upload the CSV file to this volume before proceeding!")
print(f"   1. Go to Databricks UI > Data > Volumes")
print(f"   2. Navigate to: {CATALOG} > {SCHEMA} > {VOLUME}")
print(f"   3. Click 'Upload' and upload: category_insights_data.csv")

## Step 2: Verify CSV File in Volume

In [ ]:
# List files in volume to verify CSV is uploaded
import os

volume_dir = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"

try:
    files = dbutils.fs.ls(volume_dir)
    print(f"📁 Files in volume {VOLUME}:")
    for file in files:
        print(f"   - {file.name} ({file.size} bytes)")
    
    # Check if our CSV exists
    csv_exists = any(file.name == CSV_FILE for file in files)
    if csv_exists:
        print(f"")
        print(f"✅ CSV file found: {CSV_FILE}")
    else:
        print(f"")
        print(f"❌ CSV file NOT found: {CSV_FILE}")
        print(f"   Please upload the CSV file to the volume before continuing.")
except Exception as e:
    print(f"❌ Error listing volume contents: {e}")
    print(f"   Make sure the volume exists and you have permission to access it.")

## Step 3: Read CSV Data from Volume

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, DecimalType, IntegerType

# Define schema to ensure proper data types
schema = StructType([
    StructField("Category", StringType(), False),
    StructField("Subcategory", StringType(), False),
    StructField("SKU", StringType(), False),
    StructField("ItemName", StringType(), False),
    StructField("VendorName", StringType(), False),
    StructField("YourPrice", DecimalType(10, 2), False),
    StructField("BenchmarkPrice", DecimalType(10, 2), False),
    StructField("UnitsPerYear", IntegerType(), False),
    StructField("AnnualSpend", DecimalType(12, 2), False),
    StructField("PriceIndex", DecimalType(5, 2), False),
    StructField("SavingsOpportunity", DecimalType(12, 2), False),
    StructField("CategoryStatus", StringType(), False)
])

# Read CSV from volume with explicit schema
df = spark.read \
    .format("csv") \
    .option("header", "true") \
    .schema(schema) \
    .load(VOLUME_PATH)

row_count = df.count()
print(f"✅ Loaded {row_count} rows from CSV")
display(df.limit(10))

## Step 4: Data Validation

In [ ]:
from pyspark.sql.functions import col, abs, sum as spark_sum, count, avg, round as spark_round

print("🔍 Validating data quality...")
print("")

# Check for null values
null_counts = df.select([count(col(c)).alias(c) for c in df.columns])
print("📊 Column counts (should match total rows):")
display(null_counts)

# Validate AnnualSpend calculation
validation_df = df.withColumn(
    "CalcAnnualSpend", 
    col("YourPrice") * col("UnitsPerYear")
).withColumn(
    "SpendDiff",
    abs(col("AnnualSpend") - col("CalcAnnualSpend"))
)

max_diff = validation_df.agg({"SpendDiff": "max"}).collect()[0][0]
print(f"")
print(f"✅ AnnualSpend validation: Max difference = {max_diff} (should be near 0)")

# Validate PriceIndex calculation
validation_df = validation_df.withColumn(
    "CalcPriceIndex",
    spark_round(col("YourPrice") / col("BenchmarkPrice"), 2)
).withColumn(
    "IndexDiff",
    abs(col("PriceIndex") - col("CalcPriceIndex"))
)

max_index_diff = validation_df.agg({"IndexDiff": "max"}).collect()[0][0]
print(f"✅ PriceIndex validation: Max difference = {max_index_diff} (should be near 0)")
print("")

# Summary statistics
print("📈 Summary by Category:")
summary = df.groupBy("Category").agg(
    count("*").alias("ItemCount"),
    spark_sum("AnnualSpend").alias("TotalSpend"),
    spark_sum("SavingsOpportunity").alias("TotalSavings"),
    avg("PriceIndex").alias("AvgPriceIndex")
).orderBy(col("TotalSpend").desc())

display(summary)

## Step 5: Write to Unity Catalog Delta Table

In [ ]:
# Write to Delta table
df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TARGET_TABLE)

print(f"✅ Successfully created table: {TARGET_TABLE}")
print(f"✅ Total rows: {df.count()}")

## Step 6: Verify Table Creation

In [ ]:
# Verify table structure
display(spark.sql(f"DESCRIBE EXTENDED {TARGET_TABLE}"))

In [ ]:
# Quick data preview
display(spark.sql(f"SELECT * FROM {TARGET_TABLE} LIMIT 10"))

In [ ]:
# Summary statistics
summary_query = f"""
SELECT 
  COUNT(*) as TotalRows,
  COUNT(DISTINCT Category) as UniqueCategories,
  COUNT(DISTINCT VendorName) as UniqueVendors,
  ROUND(SUM(AnnualSpend), 2) as TotalAnnualSpend,
  ROUND(SUM(SavingsOpportunity), 2) as TotalSavingsOpportunity,
  ROUND(AVG(PriceIndex), 2) as AvgPriceIndex
FROM {TARGET_TABLE}
"""

display(spark.sql(summary_query))

## Step 7: Preview Dashboard Queries

In [ ]:
# Cardiology High Variance Items (for dashboard deep dive)
cardiology_query = f"""
SELECT 
  ItemName,
  SKU,
  VendorName,
  YourPrice,
  BenchmarkPrice,
  PriceIndex,
  UnitsPerYear,
  AnnualSpend,
  SavingsOpportunity,
  CategoryStatus
FROM {TARGET_TABLE}
WHERE Category = 'Cardiology'
ORDER BY SavingsOpportunity DESC
"""

print("💊 Cardiology Deep Dive (High Variance Items):")
display(spark.sql(cardiology_query))

In [ ]:
# Category rollup for dashboard
category_query = f"""
SELECT 
  Category,
  ROUND(SUM(AnnualSpend), 2) as CategorySpend,
  ROUND(AVG(PriceIndex), 2) as CategoryPriceIndex,
  ROUND(SUM(SavingsOpportunity), 2) as CategorySavings,
  COUNT(*) as ItemCount
FROM {TARGET_TABLE}
GROUP BY Category
ORDER BY CategorySpend DESC
"""

print("📊 Category Summary:")
display(spark.sql(category_query))

In [ ]:
# Outlier analysis data (for scatter plot)
outlier_query = f"""
SELECT 
  Category,
  ROUND(SUM(AnnualSpend), 2) as TotalSpend,
  ROUND(AVG(PriceIndex), 2) as AvgPriceIndex,
  ROUND(SUM(SavingsOpportunity), 2) as TotalSavings
FROM {TARGET_TABLE}
GROUP BY Category
ORDER BY TotalSpend DESC
"""

print("🎯 Outlier Analysis (Spend vs Price Index):")
display(spark.sql(outlier_query))

## ✅ Data Load Complete!

The category insights data has been successfully loaded into Unity Catalog.  

**Table Location:** `archana_krish_fe_dsa.vizient_deep_dive.category_insights_delta`  
**Volume Location:** `/Volumes/archana_krish_fe_dsa/vizient_deep_dive/data_files/`

### Next Steps:
1. Use this table as the source for your Lakeview dashboard
2. Import the `category_insights_dashboard.json` specification
3. Create visualizations based on the queries above